<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8"><p style="margin:0 0 8px 0; color:#cba6f7; font-weight:bold; font-size:1.05em;">✅ Session 8 — Iterators &amp; Context Managers · Solutions</p><p style="margin:0;">Worked, runnable solutions for the 12 <strong>Exercises</strong> and 8 <strong>Code Challenges</strong>. Run top to bottom to verify. Try them in <code>01_iterators.ipynb</code> first.</p></div>

### Exercises — Solutions

In [ ]:
import itertools, time, io
from contextlib import contextmanager, redirect_stdout, ExitStack
from collections import deque

In [ ]:
# E1 — iter()/next()
it = iter([10, 20, 30])
print(next(it), next(it))            # 10 20

In [ ]:
# E2 — evens(n)
def evens(n):
    for i in range(n):
        yield i * 2

print(list(evens(4)))                # [0, 2, 4, 6]

In [ ]:
# E3 — Repeat (re-iterable via __iter__ generator method)
class Repeat:
    def __init__(self, v, t): self.v = v; self.t = t
    def __iter__(self):
        for _ in range(self.t):
            yield self.v

r = Repeat("x", 3)
print(list(r), list(r))              # ['x','x','x'] both times

In [ ]:
# E4 — CountUp custom iterator class
class CountUp:
    def __init__(self, start, stop): self.cur = start; self.stop = stop
    def __iter__(self): return self
    def __next__(self):
        if self.cur >= self.stop: raise StopIteration
        v = self.cur; self.cur += 1; return v

print(list(CountUp(2, 5)))           # [2, 3, 4]

In [ ]:
# E5 — infinite generator + islice
def naturals():
    n = 0
    while True:
        yield n; n += 1

print(list(itertools.islice(naturals(), 5)))   # [0, 1, 2, 3, 4]

In [ ]:
# E6 — first matching (or default)
def first(iterable, pred, default=None):
    return next((x for x in iterable if pred(x)), default)

print(first([1,3,4,6], lambda x: x%2==0), first([1,3], lambda x: x%2==0, -1))   # 4 -1

In [ ]:
# E7 — batched(iterable, n)
def batched(iterable, n):
    batch = []
    for x in iterable:
        batch.append(x)
        if len(batch) == n:
            yield batch; batch = []
    if batch:
        yield batch

print(list(batched(range(7), 3)))    # [[0,1,2],[3,4,5],[6]]

In [ ]:
# E8 — Tag context manager
class Tag:
    def __init__(self, name): self.name = name
    def __enter__(self): print(f"<{self.name}>"); return self
    def __exit__(self, *a): print(f"</{self.name}>")

with Tag("b"):
    print("hi")                      # <b> / hi / </b>

In [ ]:
# E9 — @contextmanager timer capturing elapsed
@contextmanager
def timer(out):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        out.append(time.perf_counter() - t0)

holder = []
with timer(holder):
    sum(range(1000))
print("captured:", len(holder) == 1)   # True

In [ ]:
# E10 — @contextmanager setattr_temp (set & restore)
@contextmanager
def setattr_temp(obj, attr, value):
    old = getattr(obj, attr)
    setattr(obj, attr, value)
    try:
        yield obj
    finally:
        setattr(obj, attr, old)

class C: pass
c = C(); c.x = 1
with setattr_temp(c, "x", 99):
    print("inside:", c.x)            # 99
print("restored:", c.x)              # 1

In [ ]:
# E11 — reimplement suppress(*excs)
class suppress_exc:
    def __init__(self, *excs): self.excs = excs
    def __enter__(self): return self
    def __exit__(self, et, ev, tb):
        return et is not None and issubclass(et, self.excs)

with suppress_exc(ValueError):
    raise ValueError("x")
print("suppressed, continued")

In [ ]:
# E12 — pairwise(iterable)
def pairwise(iterable):
    it = iter(iterable)
    prev = next(it, None)
    if prev is None:
        return
    for cur in it:
        yield (prev, cur); prev = cur

print(list(pairwise([1, 2, 3, 4])))  # [(1,2),(2,3),(3,4)]

### Code Challenges — Solutions

In [ ]:
# C1 — take(iterable, n)
def take(it, n): return list(itertools.islice(it, n))
print(take(range(100), 3))           # [0, 1, 2]

In [ ]:
# C2 — ilen(iterable)
def ilen(it): return sum(1 for _ in it)
print(ilen(iter([1, 2, 3, 4])))      # 4

In [ ]:
# C3 — unique(iterable), order-preserving
def unique(it):
    seen = set()
    for x in it:
        if x not in seen:
            seen.add(x); yield x

print(list(unique([1, 2, 1, 3, 2, 4])))   # [1, 2, 3, 4]

In [ ]:
# C4 — sliding_window(iterable, k)
def sliding_window(it, k):
    it = iter(it)
    win = deque(itertools.islice(it, k), maxlen=k)
    if len(win) == k:
        yield tuple(win)
    for x in it:
        win.append(x); yield tuple(win)

print(list(sliding_window([1, 2, 3, 4], 2)))   # [(1,2),(2,3),(3,4)]

In [ ]:
# C5 — reimplement enumerate
def enumerate_from(it, start=0):
    i = start
    for x in it:
        yield (i, x); i += 1

print(list(enumerate_from(["a", "b"], 1)))   # [(1,'a'), (2,'b')]

In [ ]:
# C6 — @contextmanager capture_stdout
@contextmanager
def capture_stdout():
    buf = io.StringIO()
    with redirect_stdout(buf):
        yield buf

with capture_stdout() as buf:
    print("hello")
print(repr(buf.getvalue()))          # 'hello\n'

In [ ]:
# C7 — both an iterator AND a context manager (like a file object)
class LineSource:
    def __init__(self, lines): self.lines = lines; self.closed = False
    def __enter__(self): return self
    def __exit__(self, *a): self.closed = True
    def __iter__(self): return iter(self.lines)

with LineSource(["a", "b", "c"]) as src:
    got = list(src)
print(got, "| closed:", src.closed)  # ['a','b','c'] | closed: True

In [ ]:
# C8 — dynamic context managers with ExitStack
log = []
@contextmanager
def res(name):
    log.append(f"open {name}")
    try:
        yield name
    finally:
        log.append(f"close {name}")

with ExitStack() as stack:
    for n in ["a", "b", "c"]:
        stack.enter_context(res(n))
print(log)   # open a,b,c then close c,b,a (reverse order)